<div dir="rtl" style="text-align: right; font-family: Tahoma, Arial, sans-serif; line-height: 1.9; border-right: 5px solid #20639b; padding: 8px 18px;">
  <h1 style="color:#173f5f; margin-bottom:4px;">نوت‌بوک 16: پاک‌سازی و تحلیل کاربردی با Pandas</h1>
  <p><b>سطح:</b> متوسط &nbsp; | &nbsp; <b>روش مطالعه:</b> توضیح کوتاه ← اجرای مثال ← تغییر مثال ← حل تمرین</p>
  <h3>هدف‌های یادگیری</h3>
  <ul><li>تشخیص و اصلاح دادهٔ گمشده و تکراری</li>
<li>تبدیل متن به تاریخ و عدد</li>
<li>ترکیب جدول‌ها با merge</li>
<li>ساخت pivot table و گزارش نهایی</li></ul>
  <p style="background:#eef6fb; padding:10px; border-radius:8px;">همهٔ سلول‌ها را به‌ترتیب اجرا کنید. برای یادگیری بهتر، مقدار ورودی‌ها را تغییر دهید و نتیجه را پیش‌بینی کنید.</p>
</div>

<div dir="rtl" style="text-align: right; font-family: Tahoma, Arial, sans-serif; line-height: 1.9;">
  <h2 style="color:#173f5f;">دادهٔ خام</h2>
  <p>دادهٔ واقعی معمولاً فاصلهٔ اضافی، نگارش ناهماهنگ، مقدار گمشده، عدد متنی و ردیف تکراری دارد. پاک‌سازی باید قابل توضیح و تکرارپذیر باشد.</p>
</div>

In [2]:
# Create intentionally messy sales data
import numpy as np
import pandas as pd

raw_sales = pd.DataFrame({
    "order_id": [1, 2, 3, 3, 4, 5],
    "date": ["2026-01-05", "2026/01/06", "bad-date", "bad-date", "2026-02-02", "2026-02-08"],
    "city": [" Tehran ", "shiraz", "TEHRAN", "TEHRAN", None, "Tabriz"],
    "product_id": ["P1", "P2", "P1", "P1", "P3", "P2"],
    "quantity": ["2", "3", "1", "1", "bad", "4"],
    "unit_price": [180_000, 95_000, np.nan, np.nan, 420_000, 95_000],
})

raw_sales


,order_id,date,city,product_id,quantity,unit_price
0,1,2026-01-05,Tehran,P1,2,180000.0
1,2,2026/01/06,shiraz,P2,3,95000.0
2,3,bad-date,TEHRAN,P1,1,NaN
3,3,bad-date,TEHRAN,P1,1,NaN
4,4,2026-02-02,None,P3,bad,420000.0
5,5,2026-02-08,Tabriz,P2,4,95000.0


<div dir="rtl" style="text-align: right; font-family: Tahoma, Arial, sans-serif; line-height: 1.9;">
  <h2 style="color:#173f5f;">ارزیابی کیفیت</h2>
  <p>ابتدا تعداد مقادیر گمشده، ردیف‌های تکراری و نوع ستون‌ها را ثبت کنید تا اثر پاک‌سازی قابل اندازه‌گیری باشد.</p>
</div>

In [3]:
# Profile basic data-quality issues
print("Missing values:\n", raw_sales.isna().sum())
print("Duplicate rows:", raw_sales.duplicated().sum())
print("Data types:\n", raw_sales.dtypes)


Missing values:
 order_id      0
date          0
city          1
product_id    0
quantity      0
unit_price    2
dtype: int64
Duplicate rows: 1
Data types:
 order_id        int64
date           object
city           object
product_id     object
quantity       object
unit_price    float64
dtype: object


<div dir="rtl" style="text-align: right; font-family: Tahoma, Arial, sans-serif; line-height: 1.9;">
  <h2 style="color:#173f5f;">پاک‌سازی مرحله‌ای</h2>
  <p>از کپی دادهٔ خام کار می‌کنیم، تکرار را حذف می‌کنیم، متن را استاندارد و تبدیل‌ها را با حالت امن انجام می‌دهیم. مقدار نامعتبر پس از تبدیل به NaN یا NaT تبدیل می‌شود.</p>
</div>

In [4]:
# Clean text, dates, numbers, and duplicates
sales = raw_sales.copy().drop_duplicates()

sales["city"] = (
    sales["city"]
    .astype("string")
    .str.strip()
    .str.title()
)
sales["date"] = pd.to_datetime(sales["date"], errors="coerce")
sales["quantity"] = pd.to_numeric(sales["quantity"], errors="coerce")

sales["city"] = sales["city"].fillna("Unknown")
sales["unit_price"] = sales["unit_price"].fillna(sales["unit_price"].median())
sales = sales.dropna(subset=["date", "quantity"])
sales["quantity"] = sales["quantity"].astype(int)

print(sales.isna().sum())
sales


order_id      0
date          0
city          0
product_id    0
quantity      0
unit_price    0
dtype: int64


,order_id,date,city,product_id,quantity,unit_price
0,1,2026-01-05,Tehran,P1,2,180000.0
5,5,2026-02-08,Tabriz,P2,4,95000.0


<div dir="rtl" style="text-align: right; font-family: Tahoma, Arial, sans-serif; line-height: 1.9;">
  <h2 style="color:#173f5f;">ترکیب با جدول مرجع</h2>
  <p>جدول مرجع نام و گروه محصول را دارد. <code>validate='many_to_one'</code> انتظار ما از رابطه را کنترل می‌کند.</p>
</div>

In [5]:
# Join sales with a product lookup table
products = pd.DataFrame({
    "product_id": ["P1", "P2", "P3"],
    "product_name": ["Book", "Pen Set", "Desk Lamp"],
    "category": ["Education", "Stationery", "Home"],
})

enriched = sales.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one",
)
enriched["revenue"] = enriched["quantity"] * enriched["unit_price"]
enriched["month"] = enriched["date"].dt.to_period("M").astype(str)

enriched


,order_id,date,city,product_id,quantity,unit_price,product_name,category,revenue,month
0,1,2026-01-05,Tehran,P1,2,180000.0,Book,Education,360000.0,2026-01
1,5,2026-02-08,Tabriz,P2,4,95000.0,Pen Set,Stationery,380000.0,2026-02


<div dir="rtl" style="text-align: right; font-family: Tahoma, Arial, sans-serif; line-height: 1.9;">
  <h2 style="color:#173f5f;">Pivot table</h2>
  <p>Pivot table گزارش دوبعدی می‌سازد. در این مثال درآمد شهرها در هر ماه مقایسه می‌شود.</p>
</div>

In [6]:
# Create a monthly city revenue matrix
monthly_city = pd.pivot_table(
    enriched,
    index="city",
    columns="month",
    values="revenue",
    aggfunc="sum",
    fill_value=0,
    margins=True,
)

monthly_city


month,2026-01,2026-02,All
city,,,
Tabriz,0.0,380000.0,380000.0
Tehran,360000.0,0.0,360000.0
All,360000.0,380000.0,740000.0


<div dir="rtl" style="text-align: right; font-family: Tahoma, Arial, sans-serif; line-height: 1.9;">
  <h2 style="color:#173f5f;">تمرین کوتاه</h2>
  <div style="background:#fff7df; border:1px solid #f0c36d; padding:12px; border-radius:8px;">گزارشی بر اساس category بسازید که تعداد ردیف فروش، مجموع تعداد و درآمد را نشان دهد و سهم درصدی هر گروه از درآمد کل را نیز محاسبه کند.</div>
</div>

<div dir="rtl" style="text-align: right; font-family: Tahoma, Arial, sans-serif; line-height: 1.9;">
  <h2 style="color:#173f5f;">پاسخ پیشنهادی</h2>
  <p>ابتدا پاسخ خودتان را بنویسید؛ سپس این سلول را اجرا و مقایسه کنید.</p>
</div>

In [7]:
# Report category performance and revenue share
category_report = (
    enriched.groupby("category", as_index=False)
    .agg(
        row_count=("order_id", "count"),
        units=("quantity", "sum"),
        revenue=("revenue", "sum"),
    )
    .sort_values("revenue", ascending=False)
)
category_report["revenue_share_pct"] = (
    category_report["revenue"]
    .div(category_report["revenue"].sum())
    .mul(100)
    .round(2)
)

category_report


,category,row_count,units,revenue,revenue_share_pct
1,Stationery,1,4,380000.0,51.35
0,Education,1,2,360000.0,48.65


<div dir="rtl" style="text-align: right; font-family: Tahoma, Arial, sans-serif; line-height: 1.9;">
  <h2 style="color:#173f5f;">چک‌لیست پاک‌سازی</h2>
  <p>نسخهٔ خام را نگه دارید، قواعد را مرحله‌ای اجرا کنید، تعداد ردیف قبل و بعد را مقایسه کنید، تبدیل ناموفق را بررسی کنید و رابطهٔ merge را اعتبارسنجی کنید.</p>
</div>